# RF Diffusion Implementation
https://github.com/RosettaCommons/RFdiffusion

## Setup

### Accept Terms of Service

In [ ]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/msys2

### Create Conda environment

In [ ]:
!conda env create -f ../Tools/RFdiffusion/env/SE3nv.yml

Please select the SE3nv Conda Environment from the Kernel Selector in VS Code

In [ ]:
# Note that these commands are listed but cannot be executed in the notebook directly.
# Use the kernel selector to activate conda. The next block of code will point to the folder directly
!conda activate SE3nv

### Use `pip` to set up packages

*`cd` command coes not work directly in VS code*

In [ ]:
import os
cur_dir = os.getcwd()
os.chdir('../Tools/RFdiffusion/env/SE3Transformer')

%pip install --no-cache-dir -r requirements.txt
!python setup.py install # Depricated

os.chdir(cur_dir)

### Install RFdiffusion

Does not want to run in VS Code. Can do setup with Conda terminal

In [ ]:
os.chdir('../Tools/RFdiffusion')
%pip install -e . # install the rfdiffusion module from the root of the repository

os.chdir(cur_dir)

## Use RFDiffusion

Once the environment is set up, just use the Kernel picker to use the environment

In [16]:
import os, time, subprocess

original_directory = os.getcwd()

data_path = "../Data"
rf_diff_path = "../Tools/RFdiffusion/scripts"

pdb_path = os.path.join(data_path, "TIMP3_vs_ADAM17_X_ray.pdb")
output_prefix = "../Local/rfdiffusion_output/design"


loop_insertion_site = 30
contig_string = "A1-30/6-6/A36-188" # @param {type:"string"}
num_sequences_to_generate = 50 # @param {type:"integer"}

print(original_directory)

c:\Users\ryang\Documents\Development\GitHub\PhD-Research\Generation


In [19]:
# --- Run RFdiffusion ---
if not pdb_path:
    print("Cannot run RFdiffusion without a scaffold PDB file.")
    raise Exception("No PDB File")

print("Preparing to run RFdiffusion...")

# Construct the command for RFdiffusion
run_command = [
    "python",
    os.path.join(rf_diff_path.replace('../', ''), "run_inference.py"),
    f"inference.output_prefix={output_prefix.replace('../', '')}",
    f"inference.input_pdb={pdb_path.replace('../', '')}",
    f"'contigmap.contigs=[{contig_string}]'",
    "denoiser.num_steps=50",
    f"inference.num_designs={num_sequences_to_generate}",
]

print("Running RFdiffusion to generate novel loops and structures...")
print(" ".join(run_command))

# Run the command and stream output live
os.chdir("..") 
st = time.time()
result = subprocess.run(run_command, capture_output=True, text=True)
end = time.time()
os.chdir(original_directory)

print("--- STDOUT ---")
print(result.stdout)

print("--- STDERR ---")
print(result.stderr)

print(f"RFdiffusion finished in {(end-st)/60:.2f} minutes.")

Preparing to run RFdiffusion...
Running RFdiffusion to generate novel loops and structures...
python Tools/RFdiffusion/scripts\run_inference.py inference.output_prefix=Local/rfdiffusion_output/design inference.input_pdb=Data\TIMP3_vs_ADAM17_X_ray.pdb 'contigmap.contigs=[A1-30/6-6/A36-188]' denoiser.num_steps=50 inference.num_designs=50
--- STDOUT ---

--- STDERR ---
LexerNoViableAltException: 'contigmap.contigs=[A1-30/6-6/A36-188]'
                           ^
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details


Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.

RFdiffusion finished in 0.05 minutes.


In [ ]:
# --- Parse PDB Results to Extract Sequences ---
print("parsing generated PDBs to extract sequences...")

# 3-letter to 1-letter amino acid code map
aa_map = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
        'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
        'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
        'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

generated_sequences = set()
chain_id_to_extract = contig_string.split('/')[0][0] # Get chain from contig
start_res, end_res = map(int, contig_string.split('/')[1].split('-'))
loop_length_range = range(start_res, end_res + 1)

for i in range(num_sequences_to_generate):
    pdb_file = f"{output_prefix}_{i}.pdb"
    if os.path.exists(pdb_file):
        current_sequence = []
        with open(pdb_file, 'r') as f:
            for line in f:
                if line.startswith('ATOM') and line[21] == chain_id_to_extract:
                    res_name = line[17:20]
                    if res_name in aa_map:
                        current_sequence.append(aa_map[res_name])

        # The generated loop is appended at the end of the chain in RFdiffusion outputs
        loop_seq = "".join(current_sequence[-len(current_sequence) + loop_insertion_site :])
        if len(loop_seq) in loop_length_range:
                generated_sequences.add(loop_seq)

In [ ]:
# Final cleanup
original_sequences = set(df['sequence'])
unique_new_sequences = list(generated_sequences - original_sequences)

print(f"\nExtracted {len(unique_new_sequences)} unique, novel loop sequences.")
print("Here are a few examples:")
for seq in unique_new_sequences[:5]:
    print(f"   - {seq}")